In [5]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
from datetime import datetime

mt5.initialize()
 
utc_from = datetime(2023,1,1)
utc_to = datetime(2025,8,31)

rates = mt5.copy_rates_range("EURUSDz",mt5.TIMEFRAME_D1, utc_from, utc_to)
df = pd.DataFrame(rates)
df["time"] = pd.to_datetime(df["time"], unit = 's')
df.set_index("time",inplace = True)
df

,open,high,low,close,tick_volume,spread,real_volume
time,,,,,,,
2023-01-01,1.06935,1.07020,1.06792,1.07020,983,54,0
2023-01-02,1.07017,1.07084,1.06428,1.06728,12905,15,0
2023-01-03,1.06729,1.06802,1.05197,1.05457,75923,0,0
2023-01-04,1.05458,1.06356,1.05454,1.06066,74128,0,0
2023-01-05,1.06064,1.06315,1.05152,1.05201,69614,0,0
...,...,...,...,...,...,...,...
2025-08-25,1.17080,1.17207,1.16028,1.16146,43750,0,0
2025-08-26,1.16148,1.16655,1.16021,1.16373,62443,0,0
2025-08-27,1.16373,1.16478,1.15742,1.16474,47386,0,0


In [72]:
class backtester:
    def __init__(self, df):
        self.df = df.copy()

    
    #Calculate SMA_20
    def SMA_20 (self):
        self.df['SMA_20'] = self.df['close'].rolling(20).mean()
        self.df['SMA_20'] = self.df['SMA_20'].replace(0, np.nan)
    
    #Calculate SMA_50
    def SMA_50 (self):
        self.df['SMA_50'] = self.df['close'].rolling(50).mean()
        self.df['SMA_50'] = self.df['SMA_50'].replace(0, np.nan)

    
    #Generate signal
    def trend_change(self):
        self.df["trend"] = 0
        self.df.loc[self.df['SMA_20'] > self.df['SMA_50'],"trend"] = 1 #bullish
        self.df.loc[self.df['SMA_20'] < self.df['SMA_50'],"trend"] = -1 #bearish
   
      
    #Generate signal
    def entry (self):
        #  Generate signal 2 or -2 for sell
        self.df['signal'] = self.df['trend'].diff().fillna(0)
        
        #Entry signal
        self.df['entry'] = np.nan
        self.df.loc[self.df['signal'] == 2, 'entry'] = 'buy'
        self.df.loc[self.df['signal'] == -2, 'entry'] = 'sell'
        self.df['entry'] = self.df['entry'].ffill().fillna('hold')



        #Store Position
        if 'position' not in self.df.columns:
            self.df['position'] = np.nan  # use NaN instead of 0

        # Step 1: mark entries
        self.df.loc[self.df['entry'] == 'buy', 'position'] = 1
        self.df.loc[self.df['entry'] == 'sell', 'position'] = -1

        # Step 2: forward-fill open positions
        self.df['position'] = self.df['position'].ffill().fillna(0)



        #Store entry price
        if 'entry_price' not in self.df.columns:
            self.df['entry_price'] = None

        self.df.loc[(self.df['entry'] == 'buy') & (self.df['signal'] == 2),'entry_price'] = self.df['close']
        self.df.loc[(self.df['entry'] == 'sell') & (self.df['signal'] == -2),'entry_price'] = self.df['close']
        self.df['entry_price'] = self.df['entry_price'] .replace(0, None).ffill()


    
    #Generate exit signal
    def exit (self):

        self.df['exit'] = None 
        self.df.loc[(self.df['position'].shift(1) == 1) & (self.df['signal'] == -2),'exit'] = 'exit_buy'
        self.df.loc[(self.df['position'].shift(1) == -1) & (self.df['signal'] == 2),'exit'] = 'exit_sell'

        #store exit price
        self.df.loc[self.df['exit'] == 'exit_buy','exit_price'] = self.df['close']
        self.df.loc[self.df['exit'] == 'exit_sell','exit_price'] = self.df['close']

    #def calculation of pnl
    def calculate_profit_loss(self):
        #Replace None Values 
        self.df['entry_price'] = self.df['entry_price'].replace([None], np.nan).astype(float)
        self.df['exit_price'] = self.df['exit_price'].replace([None], np.nan).astype(float)

        # Initialize
        self.df['profit_loss'] = np.nan  

        # Profit conditions
        self.df.loc[
            (self.df['entry'].shift(1) == 'buy') & (self.df['exit_price'] > self.df['entry_price'].shift(1)),
            'profit_loss'
        ] = 'profit'

        self.df.loc[
            (self.df['entry'].shift(1) == 'sell') & (self.df['exit_price'] < self.df['entry_price'].shift(1)),
            'profit_loss'
        ] = 'profit'

        # Loss conditions
        self.df.loc[
            (self.df['entry'].shift(1) == 'buy') & (self.df['exit_price'] < self.df['entry_price'].shift(1)),
            'profit_loss'
        ] = 'loss'

        self.df.loc[
            (self.df['entry'].shift(1) == 'sell') & (self.df['exit_price'] > self.df['entry_price'].shift(1)),
            'profit_loss'
        ] = 'loss'

    def metrics(self):
        #winrate
            
        wins = (self.df['profit_loss'] == 'profit').sum()
        losses = (self.df['profit_loss'] == 'loss').sum()
        total_trades = wins + losses
            
        winrate = wins/total_trades * 100 if total_trades > 0 else 0
            
        #Display metrics
        print(f''' 
            Total Trades: {total_trades}
            Wins: {wins}
            Losses: {losses}
            Winrate: {winrate}%

            ''')
        
        # Run strategy
    def run(self):
        self.SMA_20()
        self.SMA_50()
        self.trend_change()
        self.entry()
        self.exit() 
        self.calculate_profit_loss()
        self.metrics()
        return self.df

In [ ]:
bt = backtester(df)
result = bt.run()
result[['close','SMA_20','SMA_50','trend','signal','entry','entry_price','position','exit','exit_price','profit_loss']].tail(50)


 
            Total Trades: 16
            Wins: 5
            Losses: 11
            Winrate: 31.25%

            


C:\Users\Ngumo\AppData\Local\Temp\ipykernel_32112\2292998422.py:31: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'buy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  self.df.loc[self.df['signal'] == 2, 'entry'] = 'buy'
C:\Users\Ngumo\AppData\Local\Temp\ipykernel_32112\2292998422.py:56: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  self.df['entry_price'] = self.df['entry_price'] .replace(0, None).ffill()
C:\Users\Ngumo\AppData\Local\Temp\ipykernel_32112\2292998422.py:81: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'profit' has dtype incompatible with float64, plea

,close,trend,signal,entry,entry_price,position,exit,exit_price,profit_loss
time,,,,,,,,,
2025-07-03,1.17690,1,0.0,buy,1.14535,1.0,None,NaN,NaN
2025-07-04,1.17783,1,0.0,buy,1.14535,1.0,None,NaN,NaN
2025-07-06,1.17802,1,0.0,buy,1.14535,1.0,None,NaN,NaN
2025-07-07,1.17365,1,0.0,buy,1.14535,1.0,None,NaN,NaN
2025-07-08,1.17235,1,0.0,buy,1.14535,1.0,None,NaN,NaN
2025-07-09,1.17379,1,0.0,buy,1.14535,1.0,None,NaN,NaN
2025-07-10,1.17017,1,0.0,buy,1.14535,1.0,None,NaN,NaN
2025-07-11,1.16906,1,0.0,buy,1.14535,1.0,None,NaN,NaN
2025-07-13,1.16828,1,0.0,buy,1.14535,1.0,None,NaN,NaN
